=================================================================================================

In [1]:
#Consumer B
from kafka import KafkaConsumer
import psycopg2
from neo4j import GraphDatabase
import pandas as pd
import json

topic = 'transactions'

consumer = KafkaConsumer(
    topic,
    bootstrap_servers='kafka_streaming_lab:9092',
    auto_offset_reset='earliest',
    enable_auto_commit=True,
    group_id='customB-group'
)

# Neo4j polaczenie
URI = "bolt://neo4j_streaming_lab:7687"
USERNAME = "neo4j"
PASSWORD = "test1234"

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

print("Kafka-to-Neo4j processor running...")


# zapis do Neo4j
def write_tx(tx, t):

    tx.run("""
        MERGE (a:User {id: $sender})
        MERGE (b:User {id: $receiver})

        MERGE (d1:Device {id: $device_sender})
        MERGE (d2:Device {id: $device_receiver})

        MERGE (a)-[:TRANSFER {
            amount: $amount,
            timestamp: $timestamp,
            title: $title
        }]->(b)

        MERGE (a)-[:USES]->(d1)
        MERGE (b)-[:USES]->(d2)
    """,
    sender=t["sender"],
    receiver=t["receiver"],
    device_sender=t["device_sender"],
    device_receiver=t["device_receiver"],
    amount=t["amount"],
    timestamp=t["timestamp"],
    title=t["title"]
    )


try:
    for msg in consumer:

        # decode JSON
        data = json.loads(msg.value.decode('utf-8'))

        # jeśli pojedynczy dict → zamień na listę
        if isinstance(data, dict):
            data = [data]

        print(f"Received: {data}")

        with driver.session() as session:
            for t in data:
                session.execute_write(write_tx, t)

except KeyboardInterrupt:
    print("Stopping consumer...")

finally:
    consumer.close()
    driver.close()
    print("Connections closed.")

Kafka-to-Neo4j processor running...
Received: [{'sender': 'user1', 'receiver': 'user2', 'amount': 120.5, 'timestamp': '2026-05-10T10:00:00', 'device_sender': 'devA', 'device_receiver': 'devB', 'title': 'zwrot za obiad'}]
Received: [{'sender': 'user3', 'receiver': 'user4', 'amount': 75.0, 'timestamp': '2026-05-09T11:00:00', 'device_sender': 'devC', 'device_receiver': 'devD', 'title': 'oddaje pieniadze'}]
Received: [{'sender': 'user5', 'receiver': 'user6', 'amount': 200.0, 'timestamp': '2026-05-08T09:30:00', 'device_sender': 'devE', 'device_receiver': 'devF', 'title': 'zwrot kosztow'}]
Received: [{'sender': 'user2', 'receiver': 'user1', 'amount': 50.0, 'timestamp': '2026-05-07T14:00:00', 'device_sender': 'devB', 'device_receiver': 'devA', 'title': 'zwrot za paliwo'}]
Received: [{'sender': 'user7', 'receiver': 'user8', 'amount': 300.0, 'timestamp': '2026-05-06T16:00:00', 'device_sender': 'devG', 'device_receiver': 'devH', 'title': 'oddaje dlug'}]
Received: [{'sender': 'user9', 'receiver':

In [2]:
def run_cypher(driver, query: str):
    with driver.session() as session:
        result = session.run(query)
        records = list(result)
        
        if not records:
            print("Brak wyników.")
            return
        
        df = pd.DataFrame([r.data() for r in records])
        return df


def qexec(query: str):
    driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
    
    try:
        df = run_cypher(driver, query)
    finally:
        driver.close()
    
    return df

In [3]:
#Queries
q5 = """
MATCH ()-[t:TRANSFER]->()
WITH max(datetime(t.timestamp)) AS latest_date

MATCH (u1:User)-[tr1:TRANSFER]->()
WHERE datetime(tr1.timestamp) >= latest_date - duration('P2D')

MATCH (u2:User)-[tr2:TRANSFER]->()
WHERE datetime(tr2.timestamp) >= latest_date - duration('P2D')

MATCH (u1)-[:USES]->(d:Device)<-[:USES]-(u2)

WHERE id(u1) < id(u2)

RETURN DISTINCT
    d.id AS device,
    u1.id AS user1,
    u2.id AS user2
ORDER BY device
"""

q0= """
MATCH (n)
DETACH DELETE n;
"""

In [14]:
qexec(q0)

Brak wyników.


In [4]:
# Zapytania sprawdzające: (Cypher) Znajdź użytkowników korzystających z tego samego urządzenia w ciągu ostatnich 3 dni.

qexec(q5)

Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=13, column=7, offset=325>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 325, 'line': 13, 'column': 7}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH ()-[t:TRANSFER]->()\nWITH max(datetime(t.timestamp)) AS latest_date\n\nMATCH (u1:User)-[tr1:TRANSFER]->()\nWHERE datetime(tr1.timestamp) >= latest_date - duration('P2D')\n\nMATCH (u2:User)-[tr2:TRANSFER]->()\nWHERE datetime(tr2.timestamp) >= latest_date - duration('P2D')\n\nMATCH (u1)-[:USES]->(d:Device)<-[:USES]-(u2)\n\nWHERE id(u1

,device,user1,user2
0,devC,user3,user14
1,devZ,user11,user14
2,shared1,user9,user11
3,shared1,user11,user13
4,shared1,user9,user13
5,shared2,user15,user17
